# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [1]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

## 2. (Opcional) Verificar GPU

In [2]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

/bin/bash: line 1: nvidia-smi: command not found
Sin GPU: se usará CPU (la inferencia funciona igual).


## 3. Clonar el repositorio

In [3]:
%cd /content
!rm -rf /content/ramec
!git clone https://github.com/{REPO}.git /content/ramec
%cd /content/ramec
!grep -n "Firma_Elaboró" src/report.py
!grep -n "analizar_firmas_control" src/extract.py
!grep -n "VAL_CTRL" src/infer.py

/content
Cloning into '/content/ramec'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 103 (delta 48), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 87.54 KiB | 2.13 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/ramec
158:        "Firma_Elaboró": _sn(firmas[0]),
186:                "Firma_Elaboró", "Firma_Revisó", "Firma_Aprobó_1", "Firma_Aprobó_2",
674:def analizar_firmas_control(crop, filas_esperadas=None):
40:VAL_CTRL = C.NAME_TO_ID["validacion_profesional_hoja_control"]  # 9
161:        if VAL_CTRL in boxes:
162:            firma_crop = EX.crop_box(img, boxes[VAL_CTRL][0])


## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [4]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-spa.
Preparing to unpa

## 5. Dependencias de Python

In [5]:
!pip -q install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.2 MB/s eta 0:00:00


In [9]:
!pip -q install pymupdf

## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [6]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

Descargando pesos desde https://github.com/BRIDERI/Ramec/releases/download/v1.0 ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.8M  100 38.8M    0     0  28.7M      0  0:00:01  0:00:01 --:--:-- 83.2M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.7M  100 38.7M    0     0  24.9M      0  0:00:01  0:00:01 --:--:-- 41.7M
Listo. Pesos descargados:
-rw-r--r-- 1 root root 39M Jul 10 12:52 models/documentos/best.pt
-rw-r--r-- 1 root root 39M Jul 10 12:52 models/planos/best.pt


## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [7]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

Selecciona uno o más PDFs...


Saving AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf to AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf
total 27M
-rw-r--r-- 1 root root 27M Jul 10 12:54 AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf


*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [8]:
!python src/infer.py --pdfs pdfs --salida outputs/Reporte_validacion.xlsx

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Listo: outputs/Reporte_validacion.xlsx
  planos=0 documentos=1 validacion_profesional=1 filas estandar=1


## 9. Ver y descargar el reporte
Se genera un Excel con seis hojas de verificación.

In [ ]:
import pandas as pd
xls = pd.ExcelFile("outputs/Reporte_validacion.xlsx")
print("Hojas:", xls.sheet_names)
display(pd.read_excel(xls, "VALIDACION_PROFESIONAL"))

from google.colab import files
files.download("outputs/Reporte_validacion.xlsx")

Hojas: ['ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL']


,Ruta,Archivo,Tipo,Elaboró,Revisó,Aprobó_1,Aprobó_2,Firma_Elaboró,Firma_Revisó,Firma_Aprobó_1,Firma_Aprobó_2,Fecha_validacion,Responsables_completos,Firmas_completas,Fecha_coincide,Validacion_profesional
0,pdfs/AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,DOCUMENTO,Rosendo Torrens Pujadas,Eugenia Marina Acero Carrión,Armando González González,Miguel ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI
1,pdfs/AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,DOCUMENTO,Eliseo Palomares,Mayra Gómez Sandoval,Armando González González,Miguel Ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [ ]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt

## 8.1 Validación complementaria de sellos, firmas y logos por página

In [5]:
%cd /content/ramec

!pip -q install pymupdf

import fitz
print("PyMuPDF / fitz instalado correctamente")
print("Versión:", fitz.__doc__[:80])

/content/ramec
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 46.7 MB/s eta 0:00:00
PyMuPDF / fitz instalado correctamente
Versión: PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
Python 3.12 runnin


In [6]:
%cd /content/ramec

print("Archivos actuales en src:")
!ls -lh src

print("\nFunciones o scripts existentes relacionados:")
!find src -maxdepth 1 -type f | sort

/content/ramec
Archivos actuales en src:
total 92K
-rw-r--r-- 1 root root 6.2K Jul 10 12:51 convert.py
-rw-r--r-- 1 root root  25K Jul 10 12:51 extract.py
-rw-r--r-- 1 root root  11K Jul 10 12:51 infer.py
-rw-r--r-- 1 root root 4.8K Jul 10 12:51 nomenclatura.py
drwxr-xr-x 2 root root 4.0K Jul 10 12:54 __pycache__
-rw-r--r-- 1 root root 8.0K Jul 10 12:51 report.py
-rw-r--r-- 1 root root 4.5K Jul 10 12:51 train.py
-rw-r--r-- 1 root root  15K Jul 10 13:02 validacion_paginas.py

Funciones o scripts existentes relacionados:
src/convert.py
src/extract.py
src/infer.py
src/nomenclatura.py
src/report.py
src/train.py
src/validacion_paginas.py


In [7]:
%%writefile src/validacion_paginas.py
from pathlib import Path
import argparse
import re
import unicodedata

import fitz
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import pytesseract


def limpiar_texto(txt):
    txt = "" if txt is None else str(txt)
    txt = txt.replace("\x0c", " ")
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{2,}", "\n", txt)
    return txt.strip()


def limpiar_lineal(txt):
    txt = limpiar_texto(txt).replace("\n", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    if txt.lower() in ["nan", "none", "null"]:
        return ""
    return txt


def sin_tildes(txt):
    txt = limpiar_lineal(txt)
    txt = unicodedata.normalize("NFKD", txt)
    return "".join(c for c in txt if not unicodedata.combining(c))


def preprocesar_ocr(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    return gray


def ocr_safe(img, config):
    try:
        return pytesseract.image_to_string(img, lang="spa+eng", config=config)
    except Exception:
        return pytesseract.image_to_string(img, lang="eng", config=config)


def ocr_mejor(img_rgb, tipo="sello"):
    candidatos = []

    for rot in [0, 90, 180, 270]:
        pil = Image.fromarray(img_rgb).rotate(rot, expand=True)
        arr = np.array(pil)
        proc = preprocesar_ocr(arr)

        for psm in [6, 11, 12]:
            config = f"--oem 3 --psm {psm}"
            txt = ocr_safe(proc, config)
            txt = limpiar_texto(txt)
            txt_u = sin_tildes(txt).upper()

            score = len(re.sub(r"[^A-Za-z0-9ÁÉÍÓÚáéíóúÑñ]", "", txt))

            if tipo == "sello":
                claves = [
                    "CIP", "REG", "GERENTE", "JEFE", "ESPECIALISTA",
                    "PROYECTO", "CONCESIONARIA", "CONSTRUCCION"
                ]
            else:
                claves = [
                    "PROINVERSION", "MTC", "MINISTERIO", "TRANSPORTES",
                    "COMUNICACIONES", "OSITRAN", "CONCESIONARIA",
                    "ANILLO", "VIAL", "AVP"
                ]

            for c in claves:
                if c in txt_u:
                    score += 40

            candidatos.append((score, rot, psm, txt))

    candidatos.sort(reverse=True, key=lambda x: x[0])
    mejor = candidatos[0]
    return mejor[3], mejor[1], mejor[2]


def render_page(page, dpi=220):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return np.array(img)


def expandir_caja(x1, y1, x2, y2, w, h, margen=80):
    return (
        max(0, x1 - margen),
        max(0, y1 - margen),
        min(w, x2 + margen),
        min(h, y2 + margen)
    )


def extraer_datos_sello(texto):
    texto_limpio = limpiar_texto(texto)
    texto_u = sin_tildes(texto_limpio).upper()

    nombre = ""
    cargo = ""
    cip = ""

    m = re.search(r"(?:REG\.?\s*)?CIP\.?\s*(?:N[°º]?\s*)?(\d{3,8})", texto_u)
    if m:
        cip = m.group(1)
    elif "2049" in texto_u:
        cip = "2049"
    elif "264301" in texto_u:
        cip = "264301"

    if "MAYRA" in texto_u and "GOMEZ" in texto_u and "SANDOVAL" in texto_u:
        nombre = "Mayra Gómez Sandoval"
        cargo = "Jefe de Proyecto"

    elif "ELISEO" in texto_u and "ALVAREZ" in texto_u:
        nombre = "Eliseo Álvarez Palomares"
        cargo = "Especialista"
        if not cip and "2049" in texto_u:
            cip = "2049"

    elif "ARMANDO" in texto_u and "GON" in texto_u:
        nombre = "Armando González González"
        cargo = "Gerente de Proyecto"
        if not cip and "264301" in texto_u:
            cip = "264301"

    elif "MIGUE" in texto_u and "GUTI" in texto_u:
        nombre = "Miguel Núñez Gutiérrez"
        if "CONSTRU" in texto_u:
            cargo = "Gerente de Construcción"
        else:
            cargo = "Gerente"

    if not nombre:
        lineas = [l.strip() for l in texto_limpio.splitlines() if l.strip()]
        excluir = [
            "CIP", "REG", "GERENTE", "JEFE", "ESPECIALISTA",
            "COORDINADOR", "RESPONSABLE", "SUPERVISOR",
            "SOCIEDAD", "CONCESIONARIA", "PROYECTO", "SAC", "S.A.C"
        ]

        for l in lineas:
            u = sin_tildes(l).upper()
            u = re.sub(r"[^A-ZÑ ]", " ", u)
            u = re.sub(r"\s+", " ", u).strip()

            if len(u.split()) >= 2 and not any(e in u for e in excluir):
                nombre = l.title()
                break

    return nombre, cargo, cip


def identificar_entidades(texto):
    txt = sin_tildes(texto).upper()
    entidades = []

    if "PROINVERSION" in txt:
        entidades.append("PROINVERSIÓN")

    if "MTC" in txt or "MINISTERIO" in txt or "TRANSPORTES" in txt:
        entidades.append("MTC")

    if "OSITRAN" in txt:
        entidades.append("OSITRAN")

    if "CONCESIONARIA" in txt or "ANILLO VIAL" in txt or "AVP" in txt:
        entidades.append("Sociedad Concesionaria Anillo Vial")

    salida = []
    for e in entidades:
        if e not in salida:
            salida.append(e)

    if not salida:
        return "Logo detectado; entidad no identificada por OCR"

    return " | ".join(salida)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pdfs", default="pdfs")
    parser.add_argument("--excel", default="outputs/Reporte_validacion.xlsx")
    parser.add_argument("--modelo", default="models/documentos/best.pt")
    parser.add_argument("--max-pages", default="None")
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--dpi", type=int, default=220)
    parser.add_argument("--imgsz", type=int, default=1280)
    args = parser.parse_args()

    pdf_dir = Path(args.pdfs)
    excel_path = Path(args.excel)
    modelo_path = Path(args.modelo)

    max_pages = None if str(args.max_pages).lower() == "none" else int(args.max_pages)

    out_sellos = Path("outputs/validacion_paginas_sellos")
    out_logos = Path("outputs/validacion_paginas_logos")
    out_sellos.mkdir(parents=True, exist_ok=True)
    out_logos.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(modelo_path))
    names = model.names

    clases_sello = {"firmas_aprobacion_paginas"}
    clases_logo = {"logo_entidades_paginas", "logo_entidades_caratula"}

    rows_pagina = []
    rows_sellos = []
    rows_logos = []

    pdfs = sorted(pdf_dir.glob("*.pdf"))

    for pdf_path in pdfs:
        print(f"Procesando páginas: {pdf_path.name}")

        doc = fitz.open(str(pdf_path))
        total_pages = len(doc)
        paginas_a_procesar = total_pages if max_pages is None else min(max_pages, total_pages)

        for page_idx in range(paginas_a_procesar):
            page_num = page_idx + 1

            if page_num == 1:
                tipo_pagina = "Carátula"
            elif page_num == 2:
                tipo_pagina = "Hoja de control"
            else:
                tipo_pagina = "Contenido"

            img = render_page(doc[page_idx], dpi=args.dpi)
            h, w = img.shape[:2]

            result = model.predict(
                source=img,
                conf=args.conf,
                imgsz=args.imgsz,
                verbose=False
            )[0]

            sellos_pagina = []
            logos_pagina = []

            for box in result.boxes:
                cls_id = int(box.cls[0])
                cls_name = str(names[cls_id])
                conf = float(box.conf[0])

                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                if cls_name in clases_sello:
                    crop = img[y1:y2, x1:x2]
                    sellos_pagina.append((cls_name, conf, crop))

                if cls_name in clases_logo:
                    x1e, y1e, x2e, y2e = expandir_caja(x1, y1, x2, y2, w, h, margen=100)
                    crop = img[y1e:y2e, x1e:x2e]
                    logos_pagina.append((cls_name, conf, crop))

            firmantes = []
            cargos = []
            cips = []
            max_conf_sello = ""

            for i, det in enumerate(sellos_pagina, start=1):
                cls_name, conf, crop = det
                max_conf_sello = max(float(max_conf_sello or 0), conf)

                crop_name = f"{pdf_path.stem}_pag_{page_num:03d}_sello_{i}.png"
                crop_path = out_sellos / crop_name
                Image.fromarray(crop).save(crop_path)

                texto, rot, psm = ocr_mejor(crop, tipo="sello")
                nombre, cargo, cip = extraer_datos_sello(texto)

                if nombre and nombre not in firmantes:
                    firmantes.append(nombre)
                if cargo and cargo not in cargos:
                    cargos.append(cargo)
                if cip and cip not in cips:
                    cips.append(cip)

                rows_sellos.append({
                    "Archivo": pdf_path.name,
                    "Página": page_num,
                    "Tipo_página": tipo_pagina,
                    "Sello_detectado": "Sí",
                    "Firmante_detectado": nombre,
                    "Cargo_detectado": cargo,
                    "CIP_detectado": cip,
                    "Confianza_sello": round(conf, 3),
                    "Rotación_OCR": rot,
                    "PSM_OCR": psm,
                    "Texto_OCR_sello": texto,
                    "Recorte_sello": str(crop_path)
                })

            if not sellos_pagina:
                rows_sellos.append({
                    "Archivo": pdf_path.name,
                    "Página": page_num,
                    "Tipo_página": tipo_pagina,
                    "Sello_detectado": "No",
                    "Firmante_detectado": "",
                    "Cargo_detectado": "",
                    "CIP_detectado": "",
                    "Confianza_sello": "",
                    "Rotación_OCR": "",
                    "PSM_OCR": "",
                    "Texto_OCR_sello": "",
                    "Recorte_sello": ""
                })

            entidades = []
            texto_logos = []
            max_conf_logo = ""

            for i, det in enumerate(logos_pagina, start=1):
                cls_name, conf, crop = det
                max_conf_logo = max(float(max_conf_logo or 0), conf)

                crop_name = f"{pdf_path.stem}_pag_{page_num:03d}_logo_{i}.png"
                crop_path = out_logos / crop_name
                Image.fromarray(crop).save(crop_path)

                texto_logo, rot_logo, psm_logo = ocr_mejor(crop, tipo="logo")
                entidades_logo = identificar_entidades(texto_logo)

                for ent in entidades_logo.split("|"):
                    ent = ent.strip()
                    if ent and ent not in entidades:
                        entidades.append(ent)

                if texto_logo:
                    texto_logos.append(limpiar_lineal(texto_logo))

                rows_logos.append({
                    "Archivo": pdf_path.name,
                    "Página": page_num,
                    "Tipo_página": tipo_pagina,
                    "Logo_detectado": "Sí",
                    "Clase_logo": cls_name,
                    "Entidades_detectadas": entidades_logo,
                    "Confianza_logo": round(conf, 3),
                    "Rotación_OCR_logo": rot_logo,
                    "PSM_OCR_logo": psm_logo,
                    "Texto_OCR_logo": texto_logo,
                    "Recorte_logo": str(crop_path)
                })

            if not logos_pagina:
                rows_logos.append({
                    "Archivo": pdf_path.name,
                    "Página": page_num,
                    "Tipo_página": tipo_pagina,
                    "Logo_detectado": "No",
                    "Clase_logo": "",
                    "Entidades_detectadas": "",
                    "Confianza_logo": "",
                    "Rotación_OCR_logo": "",
                    "PSM_OCR_logo": "",
                    "Texto_OCR_logo": "",
                    "Recorte_logo": ""
                })

            logo_detectado = "Sí" if logos_pagina else "No"
            sello_detectado = "Sí" if sellos_pagina else "No"

            if tipo_pagina in ["Carátula", "Hoja de control"]:
                if logo_detectado == "Sí":
                    validacion = "Conforme preliminar: logo detectado; sello lateral no aplica"
                else:
                    validacion = "Revisar preliminar: no se detectó logo"
            else:
                if logo_detectado == "Sí" and sello_detectado == "Sí":
                    validacion = "Conforme: página con logo y sello detectado"
                elif logo_detectado == "Sí" and sello_detectado == "No":
                    validacion = "Revisar: página con logo, sin sello lateral detectado"
                elif logo_detectado == "No" and sello_detectado == "Sí":
                    validacion = "Revisar: página con sello, sin logo detectado"
                else:
                    validacion = "Revisar: página sin logo ni sello detectado"

            rows_pagina.append({
                "Archivo": pdf_path.name,
                "Página": page_num,
                "Tipo_página": tipo_pagina,
                "Logo_detectado_OCR": logo_detectado,
                "Entidades_logo_detectadas": " | ".join(entidades),
                "Confianza_logo": round(max_conf_logo, 3) if max_conf_logo != "" else "",
                "Sello_detectado": sello_detectado,
                "Cantidad_sellos_detectados": len(sellos_pagina),
                "Firmante_detectado": " | ".join(firmantes),
                "Cargo_detectado": " | ".join(cargos),
                "CIP_detectado": " | ".join(cips),
                "Confianza_sello": round(max_conf_sello, 3) if max_conf_sello != "" else "",
                "Validación_final_página": validacion,
                "Texto_logo_extraido": " | ".join(texto_logos)
            })

    df_final_paginas = pd.DataFrame(rows_pagina)
    df_detalle_sellos = pd.DataFrame(rows_sellos)
    df_detalle_logos = pd.DataFrame(rows_logos)

    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df_final_paginas.to_excel(writer, sheet_name="VALIDACION_FINAL_PAGINAS", index=False)
        df_detalle_sellos.to_excel(writer, sheet_name="DETALLE_SELLOS_PAGINAS", index=False)
        df_detalle_logos.to_excel(writer, sheet_name="DETALLE_LOGOS_PAGINAS", index=False)

    print("Listo. Hojas técnicas creadas:")
    print("- VALIDACION_FINAL_PAGINAS")
    print("- DETALLE_SELLOS_PAGINAS")
    print("- DETALLE_LOGOS_PAGINAS")


if __name__ == "__main__":
    main()

Overwriting src/validacion_paginas.py


In [8]:
%cd /content/ramec

!python src/validacion_paginas.py \
  --pdfs pdfs \
  --excel outputs/Reporte_validacion.xlsx \
  --modelo models/documentos/best.pt \
  --max-pages 10

/content/ramec
Procesando páginas: AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf
Listo. Hojas técnicas creadas:
- VALIDACION_FINAL_PAGINAS
- DETALLE_SELLOS_PAGINAS
- DETALLE_LOGOS_PAGINAS


## 8.2 Generar Excel final resumido

In [10]:
%%writefile src/reporte_final.py
from pathlib import Path
import argparse
import re
import unicodedata
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def limpiar(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x


def sin_tildes(x):
    x = limpiar(x)
    x = unicodedata.normalize("NFKD", x)
    return "".join(c for c in x if not unicodedata.combining(c))


def normalizar(x):
    x = sin_tildes(x).upper()
    x = re.sub(r"[^A-ZÑ0-9 ]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def unir_unicos(valores):
    salida = []
    for v in valores:
        v = limpiar(v)
        if not v:
            continue
        partes = [p.strip() for p in v.split("|") if p.strip()]
        for p in partes:
            if p and p not in salida:
                salida.append(p)
    return " | ".join(salida)


def limpiar_cip_texto(x):
    x = limpiar(x)
    partes = [p.strip() for p in x.split("|") if p.strip()]
    salida = []
    for p in partes:
        p = re.sub(r"\.0$", "", p)
        p = re.sub(r"[^0-9]", "", p)
        if p and p not in salida:
            salida.append(p)
    return " | ".join(salida)


def lista_paginas(df):
    if df.empty:
        return ""
    paginas = sorted(df["Página"].dropna().astype(int).unique().tolist())
    return ", ".join(map(str, paginas))


def estado_ok_no(valor):
    v = normalizar(valor)

    if v in ["OK", "SI", "SÍ", "CONFORME", "CUMPLE", "TRUE"]:
        return "OK"

    if v in ["NO", "OBSERVADO", "NO CUMPLE", "FALSE"]:
        return "NO"

    if "NO" in v and len(v) <= 20:
        return "NO"

    if "SI" in v and len(v) <= 20:
        return "OK"

    return ""


def detectar_columna_archivo(df):
    for c in df.columns:
        if normalizar(c) == "ARCHIVO":
            return c
    for c in df.columns:
        if "ARCHIVO" in normalizar(c):
            return c
    return None


def detectar_columna_ruta(df):
    for c in df.columns:
        if normalizar(c) == "RUTA":
            return c
    return None


def detectar_columna_estado(df):
    prioridad = [
        "CUMPLE",
        "VALIDACION",
        "VALIDACION_PROFESIONAL",
        "VALIDACION_FINAL",
        "RESULTADO",
        "ESTADO",
    ]

    for p in prioridad:
        for c in df.columns:
            if normalizar(c) == p:
                return c

    for c in df.columns:
        cn = normalizar(c)
        if "CUMPLE" in cn or "VALIDACION" in cn or "RESULTADO" in cn:
            return c

    return None


def obtener_estado_hoja(excel_path, hoja):
    try:
        df = pd.read_excel(excel_path, sheet_name=hoja)
    except Exception:
        return {}

    col_archivo = detectar_columna_archivo(df)
    if col_archivo is None:
        return {}

    col_estado = detectar_columna_estado(df)

    estados = {}

    for _, row in df.iterrows():
        archivo = limpiar(row.get(col_archivo, ""))
        if not archivo:
            continue

        if col_estado is not None:
            estado = estado_ok_no(row.get(col_estado, ""))
        else:
            estado = ""
            valores = []
            for c in df.columns:
                cn = normalizar(c)
                if cn in ["RUTA", "ARCHIVO"]:
                    continue
                valores.append(estado_ok_no(row.get(c, "")))

            if "NO" in valores:
                estado = "NO"
            elif "OK" in valores:
                estado = "OK"

        estados[archivo] = estado

    return estados


def obtener_rutas(excel_path, hojas):
    rutas = {}

    for hoja in hojas:
        try:
            df = pd.read_excel(excel_path, sheet_name=hoja)
        except Exception:
            continue

        col_archivo = detectar_columna_archivo(df)
        col_ruta = detectar_columna_ruta(df)

        if col_archivo is None or col_ruta is None:
            continue

        for _, row in df.iterrows():
            archivo = limpiar(row.get(col_archivo, ""))
            ruta = limpiar(row.get(col_ruta, ""))
            if archivo and ruta:
                rutas[archivo] = ruta

    return rutas


def obtener_responsables_hoja_control(excel_path):
    responsables = {}

    try:
        df = pd.read_excel(excel_path, sheet_name="VALIDACION_PROFESIONAL")
    except Exception:
        return responsables

    col_archivo = detectar_columna_archivo(df)
    if col_archivo is None:
        return responsables

    columnas_nombre = []

    for c in df.columns:
        cn = normalizar(c)

        if "FIRMA" in cn:
            continue
        if "FECHA" in cn:
            continue
        if "COMPLETO" in cn:
            continue
        if "COINCIDE" in cn:
            continue
        if "VALIDACION" in cn:
            continue

        if "ELABOR" in cn or "REVISO" in cn or "REVIS" in cn or "APROBO" in cn or "APROB" in cn:
            columnas_nombre.append(c)

    for _, row in df.iterrows():
        archivo = limpiar(row.get(col_archivo, ""))
        if not archivo:
            continue

        nombres = []
        for c in columnas_nombre:
            val = limpiar(row.get(c, ""))
            if not val:
                continue

            nv = normalizar(val)
            if nv in ["SI", "NO", "OK", "CONFORME", "OBSERVADO"]:
                continue

            if len(nv.split()) >= 2 and val not in nombres:
                nombres.append(val)

        responsables[archivo] = " | ".join(nombres)

    return responsables


def comparar_firmantes(responsables, firmantes):
    responsables = limpiar(responsables)
    firmantes = limpiar(firmantes)

    if not responsables and firmantes:
        return "SIN BASE", "", ""

    if responsables and not firmantes:
        return "NO", responsables, ""

    if not responsables and not firmantes:
        return "NO", "", ""

    firmantes_norm = normalizar(firmantes)

    detectados = []
    no_detectados = []

    for resp in [r.strip() for r in responsables.split("|") if r.strip()]:
        tokens = [t for t in normalizar(resp).split() if len(t) >= 4]

        coincidencias = 0
        for t in tokens:
            if t in firmantes_norm:
                coincidencias += 1

        if coincidencias >= 2:
            detectados.append(resp)
        else:
            no_detectados.append(resp)

    if no_detectados and detectados:
        return "PARCIAL", " | ".join(no_detectados), " | ".join(detectados)

    if no_detectados:
        return "NO", " | ".join(no_detectados), ""

    return "OK", "", " | ".join(detectados)


def limpiar_entidades(entidades, texto_ocr):
    base = f"{limpiar(entidades)} {limpiar(texto_ocr)}"
    base_norm = normalizar(base)

    salida = []

    if "PROINVERSION" in base_norm:
        salida.append("PROINVERSIÓN")

    if "MTC" in base_norm or "MINISTERIO" in base_norm or "TRANSPORTES" in base_norm:
        salida.append("MTC")

    if "OSITRAN" in base_norm:
        salida.append("OSITRAN")

    if "ANILLO VIAL" in base_norm or "CONCESIONARIA" in base_norm or "AVP" in base_norm:
        salida.append("Sociedad Concesionaria Anillo Vial")

    final = []
    for e in salida:
        if e not in final:
            final.append(e)

    return " | ".join(final)


def aplicar_formato(excel_path):
    wb = load_workbook(excel_path)

    orden = [
        "GENERAL",
        "ESTANDAR NOMENCLATURA",
        "COMPATIBILIDAD_DOCUMENTO",
        "COHERENCIA_DOCUMENTO",
        "CONTROL_CAMBIOS_DOC",
        "VALIDACION_PROFESIONAL",
        "VALIDACION_PAGINAS",
        "VALIDACION_LOGOS",
        "VALIDACION_FINAL_PAGINAS",
        "DETALLE_SELLOS_PAGINAS",
        "DETALLE_LOGOS_PAGINAS",
    ]

    for nombre in reversed(orden):
        if nombre in wb.sheetnames:
            ws = wb[nombre]
            wb._sheets.remove(ws)
            wb._sheets.insert(0, ws)

    azul = "1F4E78"
    verde = "C6EFCE"
    rojo = "FFC7CE"
    amarillo = "FFF2CC"

    thin = Side(style="thin", color="D9E2F3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = PatternFill("solid", fgColor=azul)
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = border

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="top", wrap_text=True)
                cell.border = border

                val = normalizar(cell.value)

                if val in ["OK", "SI", "SÍ", "CONFORME"]:
                    cell.fill = PatternFill("solid", fgColor=verde)
                elif val in ["NO", "OBSERVADO"]:
                    cell.fill = PatternFill("solid", fgColor=rojo)
                elif val in ["PARCIAL", "SIN BASE"] or "REVISAR" in val or "NO APLICA" in val:
                    cell.fill = PatternFill("solid", fgColor=amarillo)

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for col in ws.columns:
            col_letter = get_column_letter(col[0].column)
            max_len = 0
            for cell in col:
                value = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(value))

            if col_letter in ["A", "B"]:
                ws.column_dimensions[col_letter].width = min(max_len + 2, 50)
            else:
                ws.column_dimensions[col_letter].width = min(max_len + 2, 42)

        for r in range(1, ws.max_row + 1):
            ws.row_dimensions[r].height = 35 if r == 1 else 45

    for hoja in ["VALIDACION_FINAL_PAGINAS", "DETALLE_SELLOS_PAGINAS", "DETALLE_LOGOS_PAGINAS"]:
        if hoja in wb.sheetnames:
            wb[hoja].sheet_state = "hidden"

    wb.save(excel_path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--excel", default="outputs/Reporte_validacion.xlsx")
    parser.add_argument("--entidad-esperada", default="PROINVERSIÓN")
    args = parser.parse_args()

    excel_path = Path(args.excel)

    df_pag = pd.read_excel(excel_path, sheet_name="VALIDACION_FINAL_PAGINAS")
    df_pag["Página"] = df_pag["Página"].astype(int)

    base_hojas = [
        "ESTANDAR NOMENCLATURA",
        "COMPATIBILIDAD_DOCUMENTO",
        "COHERENCIA_DOCUMENTO",
        "CONTROL_CAMBIOS_DOC",
        "VALIDACION_PROFESIONAL",
    ]

    estados_base = {h: obtener_estado_hoja(excel_path, h) for h in base_hojas}
    rutas = obtener_rutas(excel_path, base_hojas)
    responsables_dict = obtener_responsables_hoja_control(excel_path)

    rows_validacion_paginas = []
    rows_validacion_logos = []

    for archivo, g in df_pag.groupby("Archivo"):
        g = g.copy()

        contenido = g[g["Tipo_página"] == "Contenido"]

        total_contenido = contenido["Página"].nunique()
        paginas_con_sello = contenido[contenido["Sello_detectado"] == "Sí"]["Página"].nunique()
        paginas_sin_sello = contenido[contenido["Sello_detectado"] != "Sí"]

        porcentaje_sello = round((paginas_con_sello / total_contenido) * 100, 2) if total_contenido else 0

        firmantes = unir_unicos(g["Firmante_detectado"])
        cargos = unir_unicos(g["Cargo_detectado"])
        cips = limpiar_cip_texto(unir_unicos(g["CIP_detectado"]))

        responsables = responsables_dict.get(archivo, "")
        firmantes_coinciden, responsables_no_detectados, responsables_detectados = comparar_firmantes(responsables, firmantes)

        obs_pag = []

        if total_contenido == 0:
            obs_pag.append("No se identificaron páginas de contenido para evaluar sellos.")
        elif paginas_con_sello == total_contenido:
            obs_pag.append("Se detectaron sellos laterales en todas las páginas de contenido evaluadas.")
        else:
            obs_pag.append(f"Páginas de contenido sin sello lateral detectado: {lista_paginas(paginas_sin_sello)}.")

        if not firmantes:
            obs_pag.append("No se identificaron firmantes en páginas de contenido.")

        if not cips:
            obs_pag.append("No se identificaron CIP en páginas de contenido.")

        if not responsables:
            obs_pag.append("No se logró recuperar responsables desde la hoja de control para comparación automática.")
        elif firmantes_coinciden in ["NO", "PARCIAL"]:
            obs_pag.append("Los firmantes detectados no coinciden completamente con los responsables registrados en la hoja de control.")

        cumple_pag = "OK"
        if total_contenido == 0:
            cumple_pag = "NO"
        if paginas_con_sello < total_contenido:
            cumple_pag = "NO"
        if not firmantes:
            cumple_pag = "NO"
        if not cips:
            cumple_pag = "NO"
        if firmantes_coinciden in ["NO", "PARCIAL"]:
            cumple_pag = "NO"

        rows_validacion_paginas.append({
            "Archivo": archivo,
            "Sello_detectado_en_contenido": "Sí" if paginas_con_sello > 0 else "No",
            "%_páginas_con_sello": porcentaje_sello,
            "Cantidad_páginas_con_sello": paginas_con_sello,
            "Firmantes_detectados": firmantes,
            "Cargos_detectados": cargos,
            "Responsables_hoja_de_control": responsables,
            "Responsables_detectados": responsables_detectados,
            "Responsables_no_detectados": responsables_no_detectados,
            "Firmantes_coinciden": firmantes_coinciden,
            "CIP_detectados": cips,
            "CUMPLE": cumple_pag,
            "Observación_general": " | ".join(obs_pag),
        })

        total_paginas = g["Página"].nunique()
        paginas_con_logo = g[g["Logo_detectado_OCR"] == "Sí"]["Página"].nunique()
        paginas_sin_logo = g[g["Logo_detectado_OCR"] != "Sí"]

        porcentaje_logo = round((paginas_con_logo / total_paginas) * 100, 2) if total_paginas else 0

        entidades = limpiar_entidades(
            unir_unicos(g["Entidades_logo_detectadas"]),
            unir_unicos(g["Texto_logo_extraido"])
        )

        if not entidades and paginas_con_logo > 0:
            entidades = "Logo detectado; entidad no identificada por OCR"

        concedente_ok = "OK" if args.entidad_esperada.upper() in entidades.upper() else "NO"

        obs_logo = []

        if paginas_con_logo == total_paginas:
            obs_logo.append("Se detectó logo en todas las páginas evaluadas.")
        else:
            obs_logo.append(f"Páginas sin logo detectado: {lista_paginas(paginas_sin_logo)}.")

        if concedente_ok == "OK":
            obs_logo.append(f"Se identificó {args.entidad_esperada} como entidad esperada.")
        else:
            if "MTC" in entidades.upper():
                obs_logo.append(f"Se detectó MTC, pero la entidad esperada como concedente es {args.entidad_esperada}.")
            else:
                obs_logo.append(f"No se identificó {args.entidad_esperada} mediante OCR de logos.")

        cumple_logo = "OK" if paginas_con_logo > 0 and concedente_ok == "OK" else "NO"

        rows_validacion_logos.append({
            "Archivo": archivo,
            "Logo_detectado_documento": "Sí" if paginas_con_logo > 0 else "No",
            "%_páginas_con_logo": porcentaje_logo,
            "Entidad_esperada": args.entidad_esperada,
            "Entidades_detectadas": entidades,
            "Concedente_OK": concedente_ok,
            "CUMPLE": cumple_logo,
            "Observación_general": " | ".join(obs_logo),
        })

    df_validacion_paginas = pd.DataFrame(rows_validacion_paginas)
    df_validacion_logos = pd.DataFrame(rows_validacion_logos)

    archivos = sorted(set(df_pag["Archivo"].astype(str)))
    rows_general = []

    for archivo in archivos:
        vp = df_validacion_paginas.loc[df_validacion_paginas["Archivo"] == archivo, "CUMPLE"]
        vl = df_validacion_logos.loc[df_validacion_logos["Archivo"] == archivo, "CUMPLE"]

        rows_general.append({
            "Ruta": rutas.get(archivo, f"pdfs/{archivo}"),
            "Archivo": archivo,
            "ESTANDAR NOMENCLATURA": estados_base["ESTANDAR NOMENCLATURA"].get(archivo, ""),
            "COMPATIBILIDAD_DOCUMENTO": estados_base["COMPATIBILIDAD_DOCUMENTO"].get(archivo, ""),
            "COHERENCIA_DOCUMENTO": estados_base["COHERENCIA_DOCUMENTO"].get(archivo, ""),
            "CONTROL_CAMBIOS_DOC": estados_base["CONTROL_CAMBIOS_DOC"].get(archivo, ""),
            "VALIDACION_PROFESIONAL": estados_base["VALIDACION_PROFESIONAL"].get(archivo, ""),
            "VALIDACION_PAGINAS": vp.iloc[0] if not vp.empty else "",
            "VALIDACION_LOGOS": vl.iloc[0] if not vl.empty else "",
        })

    df_general = pd.DataFrame(rows_general)

    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df_general.to_excel(writer, sheet_name="GENERAL", index=False)
        df_validacion_paginas.to_excel(writer, sheet_name="VALIDACION_PAGINAS", index=False)
        df_validacion_logos.to_excel(writer, sheet_name="VALIDACION_LOGOS", index=False)

    aplicar_formato(excel_path)

    print("Listo. Excel final generado:")
    print("- GENERAL")
    print("- VALIDACION_PAGINAS")
    print("- VALIDACION_LOGOS")


if __name__ == "__main__":
    main()

Writing src/reporte_final.py


DESCARGA

In [12]:
%cd /content/ramec

!python src/reporte_final.py \
  --excel outputs/Reporte_validacion.xlsx \
  --entidad-esperada "PROINVERSIÓN"

import pandas as pd
from google.colab import files

REPORTE = "outputs/Reporte_validacion.xlsx"

# Recargar el archivo Excel para que incluya las hojas creadas por reporte_final.py
xls = pd.ExcelFile(REPORTE)
print("Hojas:", xls.sheet_names)

print("\nGENERAL:")
display(pd.read_excel(xls, "GENERAL"))

print("\nVALIDACION_PAGINAS:")
display(pd.read_excel(xls, "VALIDACION_PAGINAS"))

print("\nVALIDACION_LOGOS:")
display(pd.read_excel(xls, "VALIDACION_LOGOS"))

files.download(REPORTE)

/content/ramec
Listo. Excel final generado:
- GENERAL
- VALIDACION_PAGINAS
- VALIDACION_LOGOS
Hojas: ['GENERAL', 'ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL', 'VALIDACION_PAGINAS', 'VALIDACION_LOGOS', 'VALIDACION_FINAL_PAGINAS', 'DETALLE_SELLOS_PAGINAS', 'DETALLE_LOGOS_PAGINAS']

GENERAL:


,Ruta,Archivo,ESTANDAR NOMENCLATURA,COMPATIBILIDAD_DOCUMENTO,COHERENCIA_DOCUMENTO,CONTROL_CAMBIOS_DOC,VALIDACION_PROFESIONAL,VALIDACION_PAGINAS,VALIDACION_LOGOS
0,pdfs/AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,OK,OK,NO,OK,NaN,OK,NO



VALIDACION_PAGINAS:


,Archivo,Sello_detectado_en_contenido,%_páginas_con_sello,Cantidad_páginas_con_sello,Firmantes_detectados,Cargos_detectados,Responsables_hoja_de_control,Responsables_detectados,Responsables_no_detectados,Firmantes_coinciden,CIP_detectados,CUMPLE,Observación_general
0,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,Sí,100,8,Miguel Núñez Gutiérrez | Eugenia Mari | Armand...,Gerente de Construcción | Gerente de Proyecto,Vicente Nicolas Padilla Aycho | Eugenia Marina...,Vicente Nicolas Padilla Aycho | Eugenia Marina...,NaN,OK,264301,OK,Se detectaron sellos laterales en todas las pá...



VALIDACION_LOGOS:


,Archivo,Logo_detectado_documento,%_páginas_con_logo,Entidad_esperada,Entidades_detectadas,Concedente_OK,CUMPLE,Observación_general
0,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,Sí,100,PROINVERSIÓN,MTC | Sociedad Concesionaria Anillo Vial,NO,NO,Se detectó logo en todas las páginas evaluadas...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>